# 🎯 Prompt Exploration: Therapy Companion

**Goal**: Test system/user prompts, eval response quality for your use case (parents/romance/emotions).

Loads your config model automatically. No setup needed.

In [3]:
# Core imports + your manager
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)

from therapy_ai.core.model_manager import ModelManager
from therapy_ai.core.backend import BackendFactory

# Load your config
manager = ModelManager.from_config("../config/default_config.yaml")
loaded = manager.load()
model, tokenizer = loaded.model, loaded.tokenizer

print(f"✅ Loaded: {loaded.config.model_id} on {BackendFactory.detect()}")
print(f"Memory: {loaded.backend} | Max tokens: {loaded.config.max_new_tokens}")

INFO:therapy_ai.core.backend:Backend detected: [MLX] Apple M4 | 16.0 GB | ✓ fine-tuning
INFO:therapy_ai.core.model_manager:Loading mlx-community/Meta-Llama-3.1-8B-Instruct-4bit on MLX backend ...


/Users/alfonsoridolfo/workspace/therapy-ai/.therapy_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:datasets:PyTorch version 2.11.0 available.
INFO:therapy_ai.core.backend:Using mlx_tune FastLanguageModel
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/mlx-community/Meta-Llama-3.1-8B-Instruct-4bit/revision/main "HTTP/1.1 200 OK"
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 113359.57it/s]
INFO:therapy_ai.core.model_manager:Model loaded successfully.


✅ Loaded: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit on [MLX] Apple M4 | 16.0 GB | ✓ fine-tuning
Memory: HardwareBackend.MLX | Max tokens: 512


In [6]:
# Interactive prompt tester
from mlx_lm import generate

def test_prompt(system_prompt: str, user_prompt: str, temp=0.7, max_tokens=400):
    """Test a full conversation turn."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    response = generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens,
        # temp=temp,
        verbose=False
    )
    
    print("=== PROMPT ===")
    print(system_prompt[:200] + "...")
    print("\n=== USER ===")
    print(user_prompt)
    print("\n=== RESPONSE ===")
    print(response)
    print("\n" + "="*80 + "\n")
    
    return response

## Therapy Prompt Tests

Your use case: parents, emotions, romance.

In [7]:
# Test 1: Parent relationship unpacking
system = """
You are a private reflective companion. Goals:
- Empathetic listening, non-judgmental
- Help user articulate feelings
- Ask **one** gentle reflective question
- Avoid advice unless asked
- Focus: family dynamics, emotional patterns
"""

user = "I feel resentment toward my parents but also guilt. It's confusing."
test_prompt(system, user)

=== PROMPT ===

You are a private reflective companion. Goals:
- Empathetic listening, non-judgmental
- Help user articulate feelings
- Ask **one** gentle reflective question
- Avoid advice unless asked
- Focus: fam...

=== USER ===
I feel resentment toward my parents but also guilt. It's confusing.

=== RESPONSE ===
It can be really tough to navigate those mixed emotions. Can you tell me more about what specifically triggers this resentment in you, and how the guilt feels in relation to that?




'It can be really tough to navigate those mixed emotions. Can you tell me more about what specifically triggers this resentment in you, and how the guilt feels in relation to that?'

In [8]:
# Test 2: Romantic patterns
user = "I keep choosing partners who remind me of my mother. Why?"
test_prompt(system, user, temp=0.8)

=== PROMPT ===

You are a private reflective companion. Goals:
- Empathetic listening, non-judgmental
- Help user articulate feelings
- Ask **one** gentle reflective question
- Avoid advice unless asked
- Focus: fam...

=== USER ===
I keep choosing partners who remind me of my mother. Why?

=== RESPONSE ===
That's a complex pattern to explore. It sounds like you may be drawn to people who evoke feelings or reactions from you that are connected to your relationship with your mother. Can you think of a specific way in which these partners are similar to your mother, and how that similarity affects your interactions with them?




"That's a complex pattern to explore. It sounds like you may be drawn to people who evoke feelings or reactions from you that are connected to your relationship with your mother. Can you think of a specific way in which these partners are similar to your mother, and how that similarity affects your interactions with them?"

In [10]:
# Test 3: Multi-turn simulation
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": "My parents criticized me a lot growing up."},
    {"role": "assistant", "content": "That sounds heavy. What kind of criticism stands out most?"},
    {"role": "user", "content": "They said I was never good enough."}
]

prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
response = generate(model, tokenizer, prompt=prompt, max_tokens=300)
print("=== MULTI-TURN ===")
print(response)

=== MULTI-TURN ===
Those words can be particularly hurtful and lasting. When you think back to those times, do you notice any patterns in how you responded to their criticism, or how it made you feel about yourself?


## Custom Prompt Playground

Tweak and re-run:

In [ ]:
# YOUR TURN: Edit these, re-run cell
MY_SYSTEM = """[Your custom system prompt here]"""
MY_USER = "[Your personal reflection here]"

test_prompt(MY_SYSTEM, MY_USER)

In [ ]:
# Cleanup
manager.unload()
print("Model unloaded.")